In [ ]:
# Common imports for ESBE setup-style notebooks (1/2/3/9).
# Heavy lifting lives in urbanopt_des and lib.helpers; this cell stays terse.
import shutil

from lib.helpers import (
    DEFAULT_DOCKER_IMAGE_TAG,
    select_uo_class,
    setup_notebook_paths,
)

# ── Execution mode ───────────────────────────────────────────────────────────
# Set USE_DOCKER = True  to route uo commands through the Docker container.
# Set USE_DOCKER = False to use a locally installed URBANopt CLI.
USE_DOCKER = False
UO = select_uo_class(USE_DOCKER, DEFAULT_DOCKER_IMAGE_TAG)

# Autoreload dependencies while iterating on wrapper / helper code.
%load_ext autoreload
%autoreload 2

In [ ]:
# Standard set of working-directory paths used by every notebook.
# Change this to keep models/results outside the notebook folder (recommended).
# Examples: "../esbe_2026" (one level up) or "esbe_2026" (inside this folder).
MODEL_OUTPUT_SUBDIR = "../esbe_2026"
paths = setup_notebook_paths(analysis_subdir=MODEL_OUTPUT_SUBDIR, cores_offset=2)
workdir = paths.workdir
analysis_dir = paths.analysis_dir
template_data_dir = paths.template_data_dir
num_usable_cores = paths.num_usable_cores

# update weather flag (optional)
update_weather_location = False
new_weather_epw = "USA_FL_MacDill.AFB.747880_TMY3"
new_weather_climate_zone = "1A"
weather_override = (
    (new_weather_epw, new_weather_climate_zone) if update_weather_location else None
)

### Gather the data for the OSA / PAT files


In [ ]:
# Read in the feature file from the activity_3 results and stage OSM + weather
# files into activity_2_pat in the layout PAT expects.
import json

uo_coincident = UO(
    paths.analysis_dir / "activity_3", "coincident", template_dir=template_data_dir
)
activity_pat_analysis_dir = paths.activity_dir("activity_2_pat")

feature_data = uo_coincident.project_path / "class_project_coincident.json"
run_path = uo_coincident.project_path / "run" / "baseline_scenario"

files_to_copy = []
with open(feature_data) as f:
    feature_json = json.load(f)

for feature in feature_json["features"]:
    if feature["properties"]["type"] != "Building":
        continue
    feature_id = feature["properties"]["id"]
    feature_name = feature["properties"]["name"]
    osm_file = run_path / feature_id / "in.osm"
    files_to_copy.append(
        {
            "base_dir": "seeds",
            "feature_name": feature_name,
            "file": osm_file,
            "new_filename": f"{feature_name}.osm",
        }
    )

for weather_file in (uo_coincident.project_path / "weather").glob("*"):
    files_to_copy.append(
        {
            "base_dir": "weather",
            "file": weather_file,
            "new_filename": weather_file.name,
        }
    )

# Make sure target subdirectories exist before copying.
for base_dir in {entry["base_dir"] for entry in files_to_copy}:
    (activity_pat_analysis_dir / base_dir).mkdir(parents=True, exist_ok=True)

for entry in files_to_copy:
    dest = activity_pat_analysis_dir / entry["base_dir"] / entry["new_filename"]
    print(f"Copying {entry['file']} -> {dest}")
    shutil.copy(entry["file"], dest)

# NEXT: run uo_building_to_osa.rb to convert the osa data, then drop the
# measures from that output into activity_2_pat.